In [1]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import data.cfr_data_19_23 as cfrd
import pandas as pd
import cfr.cfr_viz_helpers as vh
import plotly.express as px
import plotly.graph_objs as go
from scipy.stats import spearmanr, permutation_test

In [2]:
# Plot predictions - baseline against associated variables
# Ensure completeness on 2023 baseline data, 2023 and 2019 predictions, associations

# Load data
### Load measurement data

In [102]:
# Load AC with inferred from 2023 data, 2nd day = 2019 data

df_meas = bd.load_meas_from_excel(
    "CF_Registry_19_23_processed_with_idx", study_folder="CFR"
)

# Load AC predictions
df_res = bd.load_meas_from_excel(
    # "infer_AR_using_19_23_data_2entries_fev1_10122025",
    "infer_AR_using_19_23_data_2entries_fev1_fef2575_10122025",
    study_folder="CFR",
    str_cols_to_arrays=["Airway resistance (%)"],
).drop(columns="Healthy FEV1 (L)")
df_res23 = df_res[df_res["Date Recorded"] == datetime.date(2023, 1, 1)]
print(f"Shape: {df_res23.shape}")

# Merge predictions and baseline data
meascols = ["ID", "Date Recorded", "ecFEV1 % Predicted"]
df = df_res23.merge(df_meas[meascols], on=["ID", "Date Recorded"])
print(f"Shape dfmeas + df_res23: {df.shape}")
# CCL: Results are complete which is expected because computed based on df_meas

Shape: (1485, 3)
Shape dfmeas + df_res23: (1485, 4)


In [ ]:
# Load AC from 2019 data with 2nd day = best FEV1 (no FEF2575)
df = bd.load_meas_from_excel(
    "AR_19_data_with_best_FEV1",
    study_folder="CFR",
    str_cols_to_arrays=["Airway resistance (%)"],
)
print(f"Shape: {df.shape}")

Shape: (2046, 18)


In [4]:
df = bd.load_meas_from_excel(
    "pppfev1_ppfev1st_ppfev1ft_IV_19_plus1820_assoc",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|bFEV1, FEV1)",
        "P(HFEV1|FEV1)",
    ],
)

In [2]:
df = bd.load_meas_from_excel(
    # "infer_all_19_data_with_best_FEV1",
    "pppfev1_ppfev1st_ppfev1ft_IV_19_plus1820_assoc_p(D|M)",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|bFEV1, FEV1)",
        "P(HFEV1|FEV1)",
    ],
    use_csv=True,
    bypass_sanity_checks=True,
)

### Load IV data

In [ ]:
# Load IV data from 2019-23
# df_ass = pd.read_excel(dh.get_path_to_main() + "ExcelFiles/CFR/IV_data_19-23.xlsx")
df_ass = pd.read_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/anbitiotics_data_19-23.xlsx"
)
df_ass["IVs"] = df_ass["Home IVs"] + df_ass["Hosp IVs"]

df_avg_ivs_per_year = (
    df_ass.groupby("ID")
    .agg(
        {
            "Home IVs": "mean",
            "Hosp IVs": "mean",
            "IVs": "mean",
            "Oral": "mean",
            "Any antibiotics": "mean",
        }
    )
    .rename(
        columns={
            "Home IVs": "Avg Home IVs",
            "Hosp IVs": "Avg Hosp IVs",
            "IVs": "Avg IVs",
            "Oral": "Avg Oral",
            "Any antibiotics": "Avg Any antibiotics",
        }
    )
).reset_index()

In [11]:
df_ass = bd.load_meas_from_excel(
    "IV_data_2019",
    study_folder="CFR",
)

In [3]:
df_ass = bd.load_meas_from_excel(
    "IV_data_2018_2019_2020_aggregated", study_folder="CFR", bypass_sanity_checks=True
)

In [21]:
df_ass.head(2)

,ID,IVs,IV days,count_nan,count_zeros,raw_values,_iv_abx_bins,Date Recorded
0,B155916,2.333333,23.000000,0,1,"28, 0, 41",21-30,2019-01-01
1,B155917,0.666667,10.666667,0,2,"0, 0, 32",1-10,2019-01-01


### Create aggregated df

In [10]:
# Merge associated variables into the df
df = df.merge(df_avg_ivs_per_year, on="ID", how="left")
print(f"Shape final df: {df.shape}")

# Associations are complete

Shape final df: (2037, 27)


In [ ]:
ass_cols = ["ID", "Date Recorded", "IVs", "IV days"]
df = df.merge(df_ass[ass_cols], on=["ID", "Date Recorded"])

In [37]:
df.columns

Index(['ID', 'Age', 'Height', 'FEV1', 'FEF2575', 'best FEV1', 'Sex',
       'Date Recorded', 'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1',
       'Predicted FEV1', 'ecFEV1 % Predicted', 'FEV1 % Predicted',
       'best FEV1 old', 'idx FEV1', 'idx FEF2575%FEV1', 'idx best FEV1',
       'P(HFEV1|FEF2575, bFEV1, FEV1)', 'P(HFEV1|FEV1)',
       'Airway resistance (%)', 'IVs', 'IV days'],
      dtype='object')

In [6]:
AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, {"type": "uniform"})
AC = mh.VariableNode("Airway conductance (%)", 10, 100, 2, {"type": "uniform"})

df[AC.name] = df[AR.name].apply(lambda arr: arr[::-1])
df["mean AC"] = df[AC.name].apply(lambda ac: AC.get_mean(ac))
df.head(1)

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,P(FEV1|pred_FEV1),P(FEV1_obs|pred_FEV1),Airway resistance (%),Avg Home IVs,Avg Hosp IVs,Avg IVs,Avg Oral,Avg Any antibiotics,Airway conductance (%),mean AC
0,B155916,32,162,1.5,0.47,1.64,Female,2019-01-01,1.5,0.47,...,"[1.92074181e-247, 2.64360202e-224, 5.15354678e...",0.000005,"[8.65509846e-09, 8.15554203e-08, 6.34526903e-0...",0.2,0.4,0.6,0.8,1.4,"[1.58215462e-103, 4.6856718e-81, 2.19166211e-5...",50.223704


### Data exploration

In [95]:
df

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,FEV1%PredST,pppFEV1 - ppFEV1ST,FEV1_bin_10pct,FEV1_bin_2pct,FEV1_bin_5pct,FEV1_bin_4pct,FEV1_bin_8pct,FEV1_bin_20pct,FEV1_bin_3pct,FEV1_bin_1pct
0,B155916,32,162,1.50,0.47,1.64,Female,2019-01-01,1.50,0.47,...,48.353296,0.402135,40-50,48-50,45-50,48-52,48-56,40-60,48-51,48-49
1,B155917,45,175,2.67,1.00,2.67,Male,2019-01-01,2.67,1.00,...,69.032481,-3.557563,60-70,68-70,65-70,68-72,64-72,60-80,69-72,69-70
2,B155918,34,191,4.82,3.48,4.99,Male,2019-01-01,4.82,3.48,...,90.573232,-2.196971,90-100,90-92,90-95,88-92,88-96,80-100,90-93,90-91
3,B155921,34,150,1.44,0.59,1.45,Female,2019-01-01,1.44,0.59,...,55.258343,-0.972123,50-60,54-56,55-60,52-56,48-56,40-60,54-57,55-56
4,B155925,38,167,0.92,0.34,1.33,Female,2019-01-01,0.92,0.34,...,28.896150,0.208307,20-30,28-30,25-30,28-32,24-32,20-40,27-30,28-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2032,C222738,59,158,1.66,1.18,1.95,Female,2019-01-01,1.66,1.18,...,70.834612,-2.492822,70-80,70-72,70-75,68-72,64-72,60-80,69-72,70-71
2033,C222739,69,153,1.63,0.66,1.63,Female,2019-01-01,1.63,0.66,...,80.960138,-6.206571,80-90,80-82,80-85,80-84,80-88,80-100,78-81,80-81
2034,C222741,33,162,1.71,0.86,1.77,Female,2019-01-01,1.71,0.86,...,55.412563,-1.072575,50-60,54-56,55-60,52-56,48-56,40-60,54-57,55-56
2035,C222780,27,176,3.54,4.15,3.54,Female,2019-01-01,3.54,4.15,...,88.466165,1.919153,80-90,88-90,85-90,88-92,88-96,80-100,87-90,88-89


# Exploring associations

In [26]:
df.describe()

,Age,Height,FEV1,FEF2575,best FEV1,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,ecFEV1 % Predicted,FEV1 % Predicted,best FEV1 old,idx FEV1,idx FEF2575%FEV1,idx best FEV1,IVs,IV days
count,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000,2037.000000
mean,32.446735,168.074129,2.373108,1.665654,2.564520,2.373108,1.665654,63.547092,3.669590,64.277982,64.277982,2.564520,47.068238,31.272950,50.886107,1.579938,19.420226
std,11.520706,9.464436,0.999659,1.209953,1.005902,0.999659,1.209953,28.087835,0.693666,22.933702,22.933702,1.005902,19.990856,14.002775,20.123460,2.047869,29.679306
min,18.000000,138.000000,0.400000,0.130000,0.460000,0.400000,0.130000,8.108108,1.747108,13.313079,13.313079,0.460000,8.000000,4.000000,9.000000,0.000000,0.000000
25%,23.000000,161.000000,1.580000,0.670000,1.770000,1.580000,0.670000,40.573770,3.092335,46.710225,46.710225,1.770000,31.000000,20.000000,35.000000,0.000000,0.000000
50%,30.000000,168.000000,2.340000,1.360000,2.530000,2.340000,1.360000,58.255451,3.577940,66.020756,66.020756,2.530000,46.000000,29.000000,50.000000,0.666667,9.000000
75%,39.000000,175.000000,3.020000,2.400000,3.240000,3.020000,2.400000,81.132074,4.230593,81.742912,81.742912,3.240000,60.000000,40.000000,64.000000,2.333333,27.666667
max,80.000000,197.000000,5.530000,7.000000,5.880000,5.530000,7.000000,224.352334,5.534266,128.519504,128.519504,5.880000,110.000000,99.000000,117.000000,17.333333,287.000000


In [27]:
df.columns

Index(['ID', 'Age', 'Height', 'FEV1', 'FEF2575', 'best FEV1', 'Sex',
       'Date Recorded', 'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1',
       'Predicted FEV1', 'ecFEV1 % Predicted', 'FEV1 % Predicted',
       'best FEV1 old', 'idx FEV1', 'idx FEF2575%FEV1', 'idx best FEV1',
       'P(HFEV1|FEF2575, bFEV1, FEV1)', 'P(HFEV1|FEV1)',
       'Airway resistance (%)', 'IVs', 'IV days'],
      dtype='object')

In [5]:
import src.models.helpers as mh

HFEV1 = mh.VariableNode("Healthy FEV1 (L)", 1, 6, 0.05, prior=None)

df["mean HFEV1_pers"] = df["P(HFEV1|FEF2575, bFEV1, FEV1)"].apply(
    lambda x: HFEV1.get_mean(x)
)
df["mean HFEV1_ST"] = df["P(HFEV1|FEV1)"].apply(lambda x: HFEV1.get_mean(x))
# Prediction: Compute FEV1 in percentage of the mean personalised HFEV1 value
df["FEV1%PersPred"] = df["FEV1"] / df["mean HFEV1_pers"] * 100
# Baseline: Compute FEV1 in percentage of the mean softly truncated HFEV1 value
df["FEV1%PredST"] = df["FEV1"] / df["mean HFEV1_ST"] * 100

df["pppFEV1 - ppFEV1ST"] = df["FEV1%PersPred"] - df["FEV1%PredST"]

In [ ]:
df.to_csv(
    dh.get_path_to_main()
    + "ExcelFiles/CFR/pppfev1_ppfev1st_ppfev1ft_IV_19_plus1820_assoc.csv",
    index=False,
)

### Load data

In [9]:
df1 = bd.load_meas_from_excel(
    # "pppfev1_ppfev1_IV_19_assoc",
    "pppfev1_ppfev1_IV_19_plus1820_assoc",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|FEV1)",
    ],
)

### Ranked correlation

#### Corr of pred/baseline vs IVs

In [35]:
# challenge: find correlation in X-Y with Z where X and Y are already correlated with Z
# X-Y = 5 don't have the same meaning whether severer of mild CF
# Stratify by disease severity level (or more granular), then check if diff is correlated with IVs

# Does it corrrelated better?
iv_days = ["IV days", "Non Hosp IV days"]
iv_days = ["IV days"]
ivs = ["IVs", "Non Hosp IVs"]
ivs = ["IVs"]


def perm_test(x, y):
    """
    Returns permutation test with spearmanr statistic
    If p = 0.03: "We shuffled your data 10,000 times, and only 3% of those random shuffles produced a negative correlation
     as strong as yours. It is very unlikely this happened by luck."

    """

    def spearman_statistic(x, y):
        return spearmanr(x, y).correlation

    return permutation_test(
        (x, y),
        spearman_statistic,
        permutation_type="pairings",
        vectorized=False,
        n_resamples=10000,
        alternative="less",
    )


col = "FEV1%PersPred"
col = "FEV1%PredFT"
for ass_var in iv_days + ivs:
    x = df[col]
    y = df[ass_var]
    res1 = perm_test(x, y)
    x = df["FEV1%PredST"]
    res2 = perm_test(x, y)

    print(
        f"Spearman's ρ(pppFEV1, {ass_var}) = {res1.statistic:.3f}, one-sided p = {res1.pvalue:.2e}"
    )
    print(
        f"corr(stppFEV1,{ass_var}): {res2.statistic:.3f}, one-sided p = {res2.pvalue:.2e}"
    )

# FEV1% metrics correlated better with number of treatment days that number of treatments

Spearman's ρ(pppFEV1, IV days) = -0.532, one-sided p = 1.00e-04
corr(stppFEV1,IV days): -0.527, one-sided p = 1.00e-04
Spearman's ρ(pppFEV1, IVs) = -0.519, one-sided p = 1.00e-04
corr(stppFEV1,IVs): -0.514, one-sided p = 1.00e-04


In [ ]:
prctile = 50
t = df["pppFEV1 - ppFEV1"].abs().quantile(prctile / 100)
dftmp = df[df["pppFEV1 - ppFEV1"].abs() > t]
# iv_col = "IV days"
# title = f"ppFEV1 vs pppFEV1 coloured by {iv_col}, {dftmp.shape[0]} entries, {prctile}% biggest diff"
# fig = vh.plot_sidebyside_scatter(dftmp, "FEV1%STPred", "FEV1%PersPred", iv_col, title)

#### Stratify per ppFEV1, corr(ppPFEV1, IVs)

In [36]:
# Scatter: stratify by 10% bins (0-10, 10-20, ..., 90-100, 100+)
fev_col = "FEV1%PredST"


def binup(df, fev_col, bin_size):
    bin_edges = list(range(0, 101, bin_size)) + [float("inf")]
    bin_labels = [f"{i}-{i+bin_size}" for i in range(0, 101 - bin_size, bin_size)] + [
        f"{101 - bin_size}+"
    ]

    df[f"FEV1_bin_{bin_size}pct"] = pd.cut(
        df[fev_col],
        bins=bin_edges,
        labels=bin_labels,
        right=False,
        include_lowest=True,
    )
    return df


# df = binup(df, fev_col, 10)

In [37]:
def get_corr_for_df(df):
    # Does it corrrelated better?
    metrics = ["FEV1%PersPred", "FEV1%PredST"]
    # iv_days = ["IV days", "Non Hosp IV days"]
    iv_days = ["IV days"]

    for ass_var in iv_days:
        res1 = perm_test(df["FEV1%PersPred"], df[ass_var])
        res2 = perm_test(df["FEV1%PredST"], df[ass_var])
        print(f"corr(pppFEV1,{ass_var}): {res1.statistic:.3f} ({res1.pvalue:.3f})")
        print(f"corr(stppFEV1,{ass_var}): {res2.statistic:.3f} ({res2.pvalue:.3f})")


# df.groupby("FEV1_bin_10pct").apply(lambda group: get_corr_for_df(group))

#### Corr where diff > group bin size

In [62]:
diff = 3
dftmp = binup(df, "FEV1%PredST", diff)
dftmp = dftmp[dftmp["pppFEV1 - ppFEV1ST"].abs() > diff]
dftmp[f"FEV1_bin_{diff}pct"].value_counts()
# dftmp.groupby(f"FEV1_bin_{diff}pct").apply(lambda group: get_corr_for_df(group))

FEV1_bin_3pct
75-78    54
81-84    50
78-81    46
84-87    39
72-75    34
69-72    33
87-90    30
66-69    23
54-57    11
90-93    10
51-54     8
63-66     8
93-96     8
57-60     5
60-63     5
36-39     3
96-99     3
42-45     2
39-42     1
45-48     1
48-51     1
0-3       0
3-6       0
33-36     0
30-33     0
27-30     0
24-27     0
21-24     0
18-21     0
15-18     0
12-15     0
9-12      0
6-9       0
98+       0
Name: count, dtype: int64

In [22]:
diff = 1
dftmp = binup(df, "FEV1%PredST", diff)
dftmp = dftmp[dftmp["pppFEV1 - ppFEV1ST"].abs() > diff]
cats = list((dftmp[f"FEV1_bin_{diff}pct"].value_counts() > 30).loc[lambda s: s].index)

for cat in cats:
    print(cat, ", n:", dftmp[dftmp[f"FEV1_bin_{diff}pct"] == cat].shape[0])
    get_corr_for_df(dftmp[dftmp[f"FEV1_bin_{diff}pct"] == cat])

83-84 , n: 37
corr(pppFEV1,IV days): -0.167 (0.169)
corr(stppFEV1,IV days): 0.097 (0.710)
75-76 , n: 36
corr(pppFEV1,IV days): 0.112 (0.753)
corr(stppFEV1,IV days): 0.360 (0.983)
89-90 , n: 35
corr(pppFEV1,IV days): -0.332 (0.027)
corr(stppFEV1,IV days): -0.052 (0.389)
79-80 , n: 35
corr(pppFEV1,IV days): -0.358 (0.018)
corr(stppFEV1,IV days): -0.153 (0.200)
76-77 , n: 35
corr(pppFEV1,IV days): 0.009 (0.503)
corr(stppFEV1,IV days): -0.093 (0.302)
85-86 , n: 34
corr(pppFEV1,IV days): -0.035 (0.409)
corr(stppFEV1,IV days): -0.214 (0.119)
74-75 , n: 34
corr(pppFEV1,IV days): 0.160 (0.818)
corr(stppFEV1,IV days): 0.192 (0.863)
82-83 , n: 33
corr(pppFEV1,IV days): 0.146 (0.800)
corr(stppFEV1,IV days): 0.127 (0.769)
87-88 , n: 32
corr(pppFEV1,IV days): 0.011 (0.529)
corr(stppFEV1,IV days): 0.104 (0.708)
81-82 , n: 31
corr(pppFEV1,IV days): -0.006 (0.506)
corr(stppFEV1,IV days): -0.041 (0.407)
69-70 , n: 31
corr(pppFEV1,IV days): 0.109 (0.720)
corr(stppFEV1,IV days): -0.099 (0.304)
67-68 , n:

In [83]:
cats

['76-80', '80-84']

### Dumbel plots with IV days overlay

In [8]:
prctile = 50
# prctile = 0

metric_col = "FEV1%PersPred"
# metric_col = "FEV1%PredFT"

diff_col = "pppFEV1 - ppFEV1ST"
# diff_col = "ppFEV1FT - ppFEV1ST"

title = f"FEV1%PredST vs FEV1%PersPred, IV days, CFR 2019 (+18&20), {prctile:.0f}th prctile p(D|M)>1e-6"
# title = f"FEV1%PredST vs FEV1%PredFT ordered by pppFEV1, IV days, CFR 2019 (+18&20), {prctile:.0f}th prctile"
# title = f"FEV1%PredST vs FEV1%PredFT, IV days, CFR 2019 (+18&20), {prctile:.0f}th prctile"

df.sort_values(diff_col, inplace=True)

# dftmp = df[(df["ecFEF2575%ecFEV1"] < 70)].copy()
dftmp = df[df["P(D|M)"] > 1e-6].copy()
t = dftmp[diff_col].abs().quantile(prctile / 100)
df_to_plot = dftmp[dftmp[diff_col].abs() > t]

# Create the three dataframe masks
mask_mild = df_to_plot["FEV1%PredST"] >= 70
mask_moderate = (df_to_plot["FEV1%PredST"] >= 40) & (df_to_plot["FEV1%PredST"] < 70)
mask_severe = 40 > df_to_plot["FEV1%PredST"]

iv_col = "IV days"

fig = make_subplots(
    rows=1,
    cols=6,
    horizontal_spacing=0.04,
    column_widths=[0.18, 0.12, 0.18, 0.12, 0.18, 0.12],
    column_titles=[
        "Severe CF",
        "Severe IV days",
        "Moderate CF",
        "Moderate IV days",
        "Mild CF",
        "Mild IV days",
    ],
)


def plot_iv_days_panel(fig, dfx, iv_col, col):
    if dfx.empty:
        return
    fig.add_trace(
        go.Scatter(
            x=dfx[iv_col],
            y=dfx["ID"],
            mode="markers",
            marker=dict(color="rgba(20,20,20,0.6)", size=5),
            name=iv_col,
            showlegend=(col == 1),
            hovertemplate=f"ID: %{{y}}<br>{iv_col}: %{{x}}<extra></extra>",
        ),
        row=1,
        col=col,
    )


# 1,3,5 => dumbbell
vh.plot_scalar_dumbell(fig, df_to_plot[mask_severe], "FEV1%PredST", metric_col, col=1)
vh.plot_scalar_dumbell(fig, df_to_plot[mask_moderate], "FEV1%PredST", metric_col, col=3)
vh.plot_scalar_dumbell(fig, df_to_plot[mask_mild], "FEV1%PredST", metric_col, col=5)

# 2,4,6 => IV days
plot_iv_days_panel(fig, df_to_plot[mask_severe], iv_col, col=2)
plot_iv_days_panel(fig, df_to_plot[mask_moderate], iv_col, col=4)
plot_iv_days_panel(fig, df_to_plot[mask_mild], iv_col, col=6)

# Style dumbbell x-axes (cols 1,3,5)
for c in [1, 3, 5]:
    fig.update_xaxes(
        range=[-1, 101],
        tickvals=[0, 40, 70, 100],
        title="FEV1 % predicted",
        gridcolor="#2a3f5f",
        zeroline=True,
        zerolinecolor="#2a3f5f",
        zerolinewidth=2,
        row=1,
        col=c,
    )

# Style IV days x-axes (cols 2,4,6)
iv_max = max(1, float(df_to_plot[iv_col].max())) if not df_to_plot.empty else 1
iv_max = 200
for c in [2, 4, 6]:
    fig.update_xaxes(
        range=[-1, iv_max + 1],
        tickvals=[0, 100, 200, 300],
        title=iv_col,
        showgrid=False,
        zeroline=True,
        zerolinecolor="#2a3f5f",
        zerolinewidth=2,
        row=1,
        col=c,
    )

# Y-axes styling
for c in range(1, 7):
    fig.update_yaxes(
        showticklabels=False,
        showgrid=False,
        gridcolor="#2a3f5f",
        zeroline=True,
        zerolinecolor="#2a3f5f",
        zerolinewidth=2,
        row=1,
        col=c,
    )

fig.update_layout(
    height=1400 if prctile == 0 else 800,
    width=1500,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
# fig.show()

#### Analysis of errors in FEV1%PersPred

In [5]:
diff_col = "ppFEV1FT - ppFEV1ST"
df[df[diff_col] > 0][
    [
        "ecFEV1",
        "idx FEV1",
        "best FEV1",
        "idx best FEV1",
        diff_col,
        "FEV1%PredST",
        "FEV1%PredFT",
    ]
][-30:-1].sort_values(by="ppFEV1FT - ppFEV1ST")

,ecFEV1,idx FEV1,best FEV1,idx best FEV1,ppFEV1FT - ppFEV1ST,FEV1%PredST,FEV1%PredFT
1622,0.82,16,0.85,17,1.044143e-11,19.538862,19.538862
1640,0.79,15,0.87,17,1.053380e-11,21.695855,21.695855
1570,0.75,15,0.81,16,1.286793e-11,22.127784,22.127784
1484,0.72,14,0.78,15,2.100009e-11,28.590496,28.590496
1508,0.81,16,0.88,17,8.923351e-11,24.873507,24.873507
1485,0.75,15,0.81,16,1.398881e-10,27.573149,27.573149
1556,0.72,14,0.76,15,1.674394e-10,27.363897,27.363897
1537,0.67,13,0.72,14,1.922587e-10,29.304944,29.304944
1516,0.75,15,0.88,17,2.721485e-10,24.443621,24.443621
1731,0.75,15,0.85,17,3.022045e-10,25.425984,25.425984


In [ ]:
cols = [
    "ID",
    "Age",
    "Height",
    "best FEV1",
    "Sex",
    "ecFEV1",
    "ecFEF2575",
    "ecFEF2575%ecFEV1",
    "FEV1 % Predicted",
    "mean FEV1PersPred",
    "mean FEV1PredFT",
    "mean FEV1PredST",
    "FEV1%PersPred",
    "FEV1%PredFT",
    "FEV1%PredST",
    "pppFEV1 - ppFEV1ST",
    "ppFEV1FT - ppFEV1ST",
    "P(D|M)",
    # "IVs",
    # "IV days",
]
df[df["pppFEV1 - ppFEV1ST"] > 10][cols]

,ID,Age,Height,best FEV1,Sex,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,FEV1 % Predicted,mean FEV1PersPred,mean FEV1PredFT,mean FEV1PredST,FEV1%PersPred,FEV1%PredFT,FEV1%PredST,pppFEV1 - ppFEV1ST,ppFEV1FT - ppFEV1ST,P(D|M)
2024,B158336,18,157,2.04,Female,2.04,2.66,130.392164,66.126301,2.639204,3.035597,3.035597,77.296044,67.202590,67.202590,10.093455,0.000000,9.230332e-07
2025,B160444,23,173,2.46,Female,1.91,2.46,128.795816,50.589049,3.083201,3.714318,3.709104,61.948594,51.422636,51.494912,10.453682,-0.072277,8.664759e-08
2026,B162924,20,151,1.52,Female,1.16,1.24,106.896556,40.736187,2.222542,2.798042,2.797823,52.192481,41.457563,41.460800,10.731681,-0.003237,2.925752e-07
2027,B164056,58,165,2.16,Male,1.96,2.58,131.632647,63.173272,2.589150,3.050851,3.029469,75.700528,64.244370,64.697798,11.002731,-0.453427,8.224395e-07
2028,B162219,21,157,1.22,Female,1.10,1.15,104.545450,35.604739,2.320790,3.035319,3.035316,47.397660,36.240015,36.240045,11.157615,-0.000030,8.980497e-08
2029,C221619,34,167,1.90,Female,1.83,2.34,127.868845,54.914191,2.726115,3.272967,3.272462,67.128508,55.912565,55.921199,11.207308,-0.008635,3.088423e-07
2030,B158539,25,169,2.54,Female,2.54,3.86,151.968502,71.280956,3.015961,3.514624,3.514624,84.218601,72.269473,72.269473,11.949128,0.000000,1.705471e-07
2031,B165778,24,170,3.11,Male,2.87,5.05,175.958202,68.210721,3.455578,4.162614,4.144932,83.054129,68.947052,69.241182,13.812947,-0.294131,2.874392e-09
2032,B159700,18,161,2.35,Female,2.02,3.46,171.287132,62.075442,2.607239,3.215531,3.199609,77.476591,62.820103,63.132715,14.343876,-0.312612,2.396802e-09
2033,B169105,37,164,1.69,Female,1.37,1.71,124.817521,43.565473,2.313138,3.086866,3.086552,59.226901,44.381583,44.386101,14.840799,-0.004519,5.108203e-08


In [9]:
df[df.ID == "B162219"]

,ID,Age,Height,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,...,FEV1%PredFT,FEV1%PredST,pppFEV1 - ppFEV1ST,ppFEV1FT - ppFEV1ST,P(FEV1|M),"P(FEF2575%FEV1|M, FEV1)","P(bFEV1|M, FEV1, FEF2575%FEV1)",P(D|M),IVs,IV days
2028,B162219,21,157,1.22,Female,2019-01-01,1.1,1.15,104.54545,3.089476,...,36.240015,36.240045,11.157615,-0.00003,0.018303,0.000205,0.023933,8.980497e-08,6.333333,64.666667


In [79]:
# People that are very sick and have FEF25-75%FEV1 > 100
df[(df["ecFEF2575%ecFEV1"] > 100) & (df["FEV1 % Predicted"] < 60)][cols]

,ID,Age,Height,best FEV1,Sex,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,FEV1 % Predicted,mean FEV1PersPred,mean FEV1PredFT,mean FEV1PredST,FEV1%PersPred,FEV1%PredFT,FEV1%PredST,pppFEV1 - ppFEV1ST,ppFEV1FT - ppFEV1ST,IVs,IV days
1496,B162030,20,171,3.33,Female,2.22,2.66,119.819822,59.881868,3.632605,3.818483,3.643967,61.113161,58.138273,60.922617,0.190544,-2.784344,6.666667,55.000000
1991,B162306,35,165,2.24,Female,1.81,1.82,100.552492,56.062236,2.937397,3.179846,3.170372,61.619179,56.920986,57.091085,4.528094,-0.170100,3.666667,44.000000
1992,B161791,19,182,3.01,Female,2.41,2.55,105.809123,56.966013,3.852895,4.174764,4.157539,62.550362,57.727819,57.966988,4.583374,-0.239169,2.500000,29.000000
2003,C219729,21,168,2.17,Female,2.11,2.25,106.635076,59.158449,3.207494,3.505972,3.505462,65.783432,60.183024,60.191779,5.591654,-0.008755,0.666667,8.666667
2004,B158894,28,169,2.28,Female,1.85,1.92,103.783780,52.570945,3.124673,3.460701,3.456635,59.206203,53.457374,53.520261,5.685942,-0.062887,3.000000,46.666667
2008,B166717,29,162,1.80,Female,1.69,1.72,101.775146,52.755536,2.831758,3.146953,3.146465,59.680253,53.702737,53.711067,5.969186,-0.008330,8.333333,102.666667
2011,B157029,39,163,1.64,Female,1.60,1.64,102.499998,52.313288,2.680300,3.001246,3.001246,59.694816,53.311187,53.311187,6.383628,0.000000,2.000000,13.666667
2014,C220745,39,154,1.52,Female,1.40,1.52,108.571429,51.634619,2.320370,2.660985,2.660568,60.335192,52.612096,52.620347,7.714845,-0.008251,0.666667,9.333333
2016,B160766,23,185,3.14,Male,2.67,3.14,117.602997,52.415924,4.328392,4.954793,4.952901,61.685733,53.887216,53.907802,7.777930,-0.020586,0.000000,0.000000
2017,B169855,20,161,2.16,Female,1.75,2.09,119.428567,53.640163,2.802348,3.211125,3.205773,62.447624,54.498039,54.589023,7.858601,-0.090984,3.666667,39.333333


In [84]:
df[(df["ecFEF2575%ecFEV1"] < 100) & (df["FEV1 % Predicted"] > 60)][cols]
df[(df["ecFEF2575%ecFEV1"] < 100)][cols]

,ID,Age,Height,best FEV1,Sex,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,FEV1 % Predicted,mean FEV1PersPred,mean FEV1PredFT,mean FEV1PredST,FEV1%PersPred,FEV1%PredFT,FEV1%PredST,pppFEV1 - ppFEV1ST,ppFEV1FT - ppFEV1ST,IVs,IV days
0,B162543,20,167,4.18,Female,3.01,2.88,95.681067,85.371516,4.306868,4.323308,3.577866,69.888380,69.622612,84.128363,-14.239983,-14.505751,0.333333,4.333333
1,B163584,60,161,2.81,Female,1.95,1.07,54.871796,79.690253,2.992444,2.982316,2.471380,65.164135,65.385438,78.903299,-13.739164,-13.517861,2.000000,14.333333
2,B158415,35,186,5.88,Male,4.42,4.21,95.248868,91.704893,5.907549,5.909184,5.001867,74.819523,74.798818,88.367010,-13.547486,-13.568192,0.000000,0.000000
4,B158729,20,180,5.67,Male,4.55,4.38,96.263735,95.200989,5.783340,5.788057,5.042917,78.674261,78.610143,90.225562,-11.551301,-11.615419,0.500000,3.500000
5,B164616,18,175,4.86,Male,4.06,2.42,59.605914,92.417127,5.154843,5.118314,4.585510,78.760892,79.322988,88.539776,-9.778884,-9.216789,1.333333,7.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1979,B164052,39,158,2.15,Female,1.36,1.28,94.117644,47.504037,2.594724,2.840243,2.809075,52.414047,47.883230,48.414506,3.999541,-0.531276,13.666667,264.000000
1982,B158673,42,157,1.81,Female,1.72,1.71,99.418605,62.389357,2.544345,2.709913,2.706478,67.600904,63.470668,63.551235,4.049669,-0.080568,0.000000,0.000000
1986,B164800,18,154,1.77,Male,1.71,1.58,92.397661,51.695033,3.015408,3.252994,3.252923,56.708744,52.566957,52.568112,4.140631,-0.001155,0.000000,0.000000
1990,B163971,40,167,2.17,Male,2.00,1.91,95.499998,54.411650,3.329260,3.601625,3.600091,60.073407,55.530487,55.554157,4.519250,-0.023670,3.333333,64.666667


In [10]:
df[df.ID == "B162153"]

,ID,Age,Height,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,...,FEV1%PredFT,FEV1%PredST,pppFEV1 - ppFEV1ST,ppFEV1FT - ppFEV1ST,P(FEV1|M),"P(FEF2575%FEV1|M, FEV1)","P(bFEV1|M, FEV1, FEF2575%FEV1)",P(D|M),IVs,IV days
9,B162153,28,168,3.65,Female,2019-01-01,3.03,3.08,101.650163,3.475033,...,77.384712,85.49815,-7.423371,-8.113438,0.013406,0.032494,0.004049,0.000002,0.0,0.0


In [32]:
df[(df[diff_col].abs() > 0) & df[diff_col].abs() < 1]

,ID,Age,Height,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,...,mean FEV1PersPred,mean FEV1PredFT,mean FEV1PredST,FEV1%PersPred,FEV1%PredFT,FEV1%PredST,pppFEV1 - ppFEV1ST,ppFEV1FT - ppFEV1ST,IVs,IV days
1096,B169504,49,174,3.56,Male,2019-01-01,3.56,3.16,88.764049,3.776470,...,4.033575,4.011228,4.011228,88.259177,88.750876,88.750876,-0.491698,0.0,0.000000,0.000000
1097,B162555,30,167,3.68,Male,2019-01-01,3.68,3.25,88.315216,3.906308,...,4.137991,4.115474,4.115474,88.932057,89.418618,89.418618,-0.486561,0.0,0.000000,0.000000
1098,B171921,18,161,3.59,Female,2019-01-01,3.59,3.48,96.935936,3.254105,...,3.782410,3.763153,3.763153,94.913033,95.398735,95.398735,-0.485702,0.0,0.000000,0.000000
1099,B162486,24,179,2.31,Male,2019-01-01,2.31,0.94,40.692642,4.717994,...,4.674411,4.629212,4.629212,49.417986,49.900504,49.900504,-0.482518,0.0,2.000000,29.333333
831,B169743,66,163,1.58,Female,2019-01-01,1.58,1.08,68.354431,2.339089,...,2.336070,2.295546,2.295546,67.634958,68.828957,68.828957,-1.193998,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,B159774,26,175,3.44,Male,2019-01-01,3.44,1.98,57.558139,4.441118,...,4.588728,4.409475,4.409475,74.966305,78.013817,78.013817,-3.047512,0.0,0.000000,0.000000
1917,B157883,36,161,2.59,Male,2019-01-01,2.55,2.40,94.117653,3.477750,...,3.344987,3.438902,3.438902,76.233481,74.151581,74.151581,2.081900,0.0,0.000000,0.000000
277,B164057,34,165,2.64,Male,2019-01-01,2.64,1.31,49.621208,3.715842,...,3.823341,3.660946,3.660946,69.049566,72.112511,72.112511,-3.062945,0.0,0.000000,0.000000
1750,B164402,32,164,3.23,Male,2019-01-01,3.23,3.05,94.427243,3.708350,...,3.761467,3.786510,3.786510,85.870763,85.302827,85.302827,0.567936,0.0,0.333333,4.666667


#### Explore limits of the model P(D|M)

In [22]:
import plotly.express as px

fig = px.histogram(df, "P(D|M)")
fig.update_xaxes(nticks=50)
fig.show()

In [18]:
df[df["P(D|M)"] > 10e-6].sort_values(by=["pppFEV1 - ppFEV1ST"])

,ID,Age,Height,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,...,mean FEV1PredST,FEV1%PersPred,FEV1%PredFT,FEV1%PredST,pppFEV1 - ppFEV1ST,ppFEV1FT - ppFEV1ST,P(FEV1|M),"P(FEF2575%FEV1|M, FEV1)","P(bFEV1|M, FEV1, FEF2575%FEV1)",P(D|M)
66,B168731,80,155,1.71,Female,2019-01-01,1.55,1.14,73.548388,1.747108,...,1.866711,78.226777,79.278497,83.033742,-4.806965,-3.755245,0.020955,0.021899,0.023180,0.000011
112,B171501,64,153,1.70,Female,2019-01-01,1.55,0.88,56.774195,2.096030,...,2.087913,70.122183,72.628762,74.236814,-4.114631,-1.608052,0.025048,0.016862,0.024536,0.000010
148,B158623,75,159,1.48,Female,2019-01-01,1.48,0.87,58.783783,1.971433,...,1.971763,71.271399,75.059727,75.059727,-3.788328,0.000000,0.026018,0.018942,0.026687,0.000013
164,B156472,68,166,1.84,Female,2019-01-01,1.57,0.83,52.866239,2.370384,...,2.323252,63.895950,66.039184,67.577682,-3.681732,-1.538498,0.023397,0.019798,0.022217,0.000010
406,C222738,59,158,1.95,Female,2019-01-01,1.66,1.18,71.084336,2.377677,...,2.343487,68.341789,68.617558,70.834612,-2.492822,-2.217054,0.023115,0.030574,0.021267,0.000015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1929,B162297,29,138,1.13,Female,2019-01-01,1.05,0.86,81.904767,2.279890,...,2.239220,49.335092,46.890658,46.891335,2.443756,-0.000677,0.024810,0.022578,0.026103,0.000015
1930,B161384,33,143,1.85,Female,2019-01-01,1.79,1.80,100.558659,2.411703,...,2.384863,77.505691,74.611399,75.056725,2.448967,-0.445326,0.022906,0.025494,0.023155,0.000014
1958,C219298,30,152,2.02,Female,2019-01-01,2.02,2.04,100.990098,2.786120,...,2.752331,76.511569,73.392325,73.392325,3.119245,0.000000,0.019916,0.024843,0.020922,0.000010
1961,B162985,53,161,1.92,Female,2019-01-01,1.92,1.90,98.958334,2.634055,...,2.605168,76.931128,73.699673,73.699673,3.231455,0.000000,0.020720,0.026508,0.022056,0.000012


### More granular plot and corr than basic CF severity

In [38]:
prctile = 50
# prctile = 0

title = f"FEV1%PredST vs FEV1%PersPred, IV days, CFR 2019 (+18&20), 2 moderate, {prctile:.0f}th prctile"

diff_col = "pppFEV1 - ppFEV1ST"
df.sort_values(diff_col, inplace=True)
t = df[diff_col].abs().quantile(prctile / 100)
df_to_plot = df[df[diff_col].abs() > t]

# Severity masks (5 groups)
mask_mild1 = df_to_plot["FEV1%PredST"] >= 90
mask_mild2 = (df_to_plot["FEV1%PredST"] >= 80) & (df_to_plot["FEV1%PredST"] < 90)
mask_mild3 = (df_to_plot["FEV1%PredST"] >= 70) & (df_to_plot["FEV1%PredST"] < 80)
mask_moderate1 = (df_to_plot["FEV1%PredST"] >= 50) & (df_to_plot["FEV1%PredST"] < 70)
mask_moderate2 = (df_to_plot["FEV1%PredST"] >= 40) & (df_to_plot["FEV1%PredST"] < 50)
mask_severe = df_to_plot["FEV1%PredST"] < 40

iv_col = "IV days"

groups = [
    ("Severe (<40)", mask_severe),
    ("Moderate 2 (40-69)", mask_moderate2 | mask_moderate1),
    # ("Moderate 1 (50-69)", mask_moderate1),
    ("Mild 2 (70-79)", mask_mild3),
    ("Mild 2 (80-89)", mask_mild2),
    ("Mild 1 (>=90)", mask_mild1),
]

# 5 x 2 layout => 10 columns
# fig = make_subplots(
#     rows=1,
#     cols=10,
#     horizontal_spacing=0.02,
#     column_widths=[0.11, 0.09] * 5,
#     column_titles=[x for name, _ in groups for x in (name, f"{name} IV days")],
# )


# def plot_iv_days_panel(fig, dfx, iv_col, col):
#     if dfx.empty:
#         return
#     fig.add_trace(
#         go.Scatter(
#             x=dfx[iv_col],
#             y=dfx["ID"],
#             mode="markers",
#             marker=dict(color="rgba(20,20,20,0.6)", size=5),
#             name=iv_col,
#             showlegend=(col == 2),
#             hovertemplate=f"ID: %{{y}}<br>{iv_col}: %{{x}}<extra></extra>",
#         ),
#         row=1,
#         col=col,
#     )


# # Plot each pair of columns: odd=dumbbell, even=IV days
# for i, (_, mask) in enumerate(groups):
#     dumbbell_col = 2 * i + 1
#     ivdays_col = 2 * i + 2

#     vh.plot_scalar_dumbell(
#         fig, df_to_plot[mask], "FEV1%PredST", "FEV1%PersPred", col=dumbbell_col
#     )
#     plot_iv_days_panel(fig, df_to_plot[mask], iv_col, col=ivdays_col)

# # Style dumbbell x-axes (odd columns)
# for c in [1, 3, 5, 7, 9]:
#     fig.update_xaxes(
#         range=[-1, 101],
#         tickvals=[0, 40, 70, 100],
#         title="FEV1 % predicted",
#         gridcolor="#2a3f5f",
#         zeroline=True,
#         zerolinecolor="#2a3f5f",
#         zerolinewidth=2,
#         row=1,
#         col=c,
#     )

# # Style IV days x-axes (even columns)
# iv_max = max(1, float(df_to_plot[iv_col].max())) if not df_to_plot.empty else 1
# iv_max = max(200, iv_max)
# for c in [2, 4, 6, 8, 10]:
#     fig.update_xaxes(
#         range=[-1, iv_max + 1],
#         tickvals=[0, 100, 200, 300],
#         title=iv_col,
#         showgrid=False,
#         zeroline=True,
#         zerolinecolor="#2a3f5f",
#         zerolinewidth=2,
#         row=1,
#         col=c,
#     )

# # Y-axis styling
# for c in range(1, 11):
#     fig.update_yaxes(
#         showticklabels=False,
#         showgrid=False,
#         gridcolor="#2a3f5f",
#         zeroline=True,
#         zerolinecolor="#2a3f5f",
#         zerolinewidth=2,
#         row=1,
#         col=c,
#     )

# fig.update_layout(
#     height=1400 if prctile == 0 else 900,
#     width=2400,
#     font=dict(size=12),
#     showlegend=True,
#     title=title,
#     plot_bgcolor="white",
#     paper_bgcolor="white",
# )

# fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
# fig.show()

In [ ]:
for group_name, group_mask in groups:
    dfg = df_to_plot.loc[group_mask].copy()

    print(f"\n{group_name} (n={len(dfg)})")
    if len(dfg) < 3:
        print("Not enough data to compute reliable Spearman correlations.")
        continue

    res_st = perm_test(dfg["FEV1%PredST"], dfg["IV days"])
    res_ft = perm_test(dfg["FEV1%PredFT"], dfg["IV days"])
    res_pers = perm_test(dfg["FEV1%PersPred"], dfg["IV days"])

    print(
        f"spearmanr(FEV1%PredST, IV days):   r={res_st.correlation:.3f}, p={res_st.pvalue:.3g}"
    )
    print(
        f"spearmanr(FEV1%PredFT, IV days):   r={res_st.correlation:.3f}, p={res_st.pvalue:.3g}"
    )
    print(
        f"spearmanr(FEV1%PersPred, IV days): r={res_pers.correlation:.3f}, p={res_pers.pvalue:.3g}"
    )


Severe CF (<40) (n=9)
spearmanr(FEV1%PredST, IV days):   r=0.201, p=0.604
spearmanr(FEV1%PersPred, IV days): r=0.385, p=0.306

Moderate CF (40-69) (n=332)
spearmanr(FEV1%PredST, IV days):   r=-0.232, p=1.9e-05
spearmanr(FEV1%PersPred, IV days): r=-0.168, p=0.00218

Mild 3 (70-79) (n=298)
spearmanr(FEV1%PredST, IV days):   r=0.025, p=0.661
spearmanr(FEV1%PersPred, IV days): r=-0.027, p=0.637

Mild 2 (80-89) (n=300)
spearmanr(FEV1%PredST, IV days):   r=-0.049, p=0.397
spearmanr(FEV1%PersPred, IV days): r=-0.062, p=0.281

Mild 1 (>=90) (n=79)
spearmanr(FEV1%PredST, IV days):   r=-0.188, p=0.0977
spearmanr(FEV1%PersPred, IV days): r=-0.107, p=0.349


In [11]:
import numpy as np
import pandas as pd
from scipy import stats


def perm_test(x, y):
    """
    Wrapper for permutation test using Spearman correlation.
    Returns the result object which contains .statistic and .pvalue
    """

    def spearman_stat(a, b):
        # We use [0] to get just the correlation coefficient
        return stats.spearmanr(a, b).correlation

    return stats.permutation_test(
        (x, y),
        spearman_stat,
        permutation_type="pairings",
        alternative="less",
        n_resamples=5000,
    )


# --- Configuration ---
diff_col = "pppFEV1 - ppFEV1ST"
# diff_col = "ppFEV1FT - ppFEV1ST"
iv_col = "IV days"

# Define the groups and their specific percentile lists
# Severe gets the 50-100% range, others get the 10-100% range
group_configs = [
    ("Severe (<40)", df["FEV1%PredST"] < 40, [60, 70, 80, 90]),
    (
        "Moderate 2 (40-49)",
        (df["FEV1%PredST"] >= 40) & (df["FEV1%PredST"] < 50),
        [0, 25, 50, 75, 90],
    ),
    (
        "Moderate 1 (50-69)",
        (df["FEV1%PredST"] >= 50) & (df["FEV1%PredST"] < 70),
        [0, 25, 50, 75, 90],
    ),
    (
        "Mild 3 (70-79)",
        (df["FEV1%PredST"] >= 70) & (df["FEV1%PredST"] < 80),
        [0, 25, 50, 75, 90],
    ),
    (
        "Mild 2 (80-89)",
        (df["FEV1%PredST"] >= 80) & (df["FEV1%PredST"] < 90),
        [0, 25, 50, 75, 90],
    ),
    ("Mild 1 (>=90)", df["FEV1%PredST"] >= 90, [0, 25, 50, 75, 90]),
]

# --- Execution ---
for group_name, mask, percentiles in group_configs:
    print(f"\n{'='*40}\nGROUP: {group_name}\n{'='*40}")

    # Isolate the severity group first
    df_group = df[mask].copy()

    if df_group.empty:
        print("No data in this severity category.")
        continue

    for prctile in percentiles:
        # Calculate the threshold for this specific group's diff column
        # Use (100 - prctile) because if user wants "90% diff",
        # they usually mean the top 10% of differences.
        # If you mean "diff > the 90th percentile value", use (prctile / 100)
        t = df_group[diff_col].abs().quantile(prctile / 100)

        # Filter: keeping only those with high discrepancy
        df_sub = df_group[df_group[diff_col].abs() >= t].copy()

        n = len(df_sub)
        print(
            f"\n-- Percentile Threshold: {prctile}% (Threshold value: {t:.2f}, n={n})"
        )

        if n < 5:  # Permutation tests need a tiny bit of data to shuffle
            print("   >>> Skipping: Sample size too small.")
            continue

        # Run Tests
        res_st = perm_test(df_sub["FEV1%PredST"], df_sub[iv_col])
        res_pers = perm_test(df_sub["FEV1%PersPred"], df_sub[iv_col])

        # Print results with clear formatting
        print(f"   ST Correlation:   r={res_st.statistic:.3f} | p={res_st.pvalue:.4f}")
        print(
            f"   Pers Correlation: r={res_pers.statistic:.3f} | p={res_pers.pvalue:.4f}"
        )


GROUP: Severe (<40)

-- Percentile Threshold: 60% (Threshold value: 0.28, n=137)


/Applications/anaconda3/envs/phd/lib/python3.10/site-packages/scipy/stats/_resampling.py:1492: RuntimeWarning:

overflow encountered in scalar power



   ST Correlation:   r=-0.274 | p=0.0010
   Pers Correlation: r=-0.272 | p=0.0008

-- Percentile Threshold: 70% (Threshold value: 0.31, n=103)


/Applications/anaconda3/envs/phd/lib/python3.10/site-packages/scipy/stats/_resampling.py:1492: RuntimeWarning:

overflow encountered in scalar power



KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats


def perm_test(x, y):
    """
    Wrapper for permutation test using Spearman correlation.
    Returns the result object which contains .statistic and .pvalue
    """

    def spearman_stat(a, b):
        # We use [0] to get just the correlation coefficient
        return stats.spearmanr(a, b).correlation

    return stats.permutation_test(
        (x, y),
        spearman_stat,
        permutation_type="pairings",
        alternative="less",
        n_resamples=5000,
    )


def corr_diff(x, y, z, n_bootstrap=10000, random_state=None):
    """
    Bootstrapped differences in spearman ranked correlations
    It's as if you were sampling from the true distrubtion of corr differences
    If the X% CI interval doesn't have 0 in it, then it's significant
    """
    np.random.seed(random_state)
    n = len(x)

    # x should be more negative
    corr_x_obs, _ = stats.spearmanr(x, z, alternative="less")
    corr_y_obs, _ = stats.spearmanr(x, z, alternative="less")
    corr_diff_obs = corr_x_obs - corr_y_obs

    bootstrapped_diff = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, replace=True)
        corr_x_b, _ = stats.spearmanr(x[idx], alternative="less")
        corr_y_b, _ = stats.spearmanr(y[idx], alternative="less")
        bootstrapped_diff.append(corr_x_b - corr_y_b)

    # get CI range
    bootstrap_diffs = np.array(bootstrap_diffs)
    prctile_5 = np.percentile(bootstrap_diffs, 5)
    
    test = prctile_5 < 0


# --- Configuration ---
diff_col = "pppFEV1 - ppFEV1ST"
# diff_col = "ppFEV1FT - ppFEV1ST"
# metric = "FEV1%PredFT"
metric = "FEV1%PersPred"
iv_col = "IV days"

df3 = df[df["P(D|M)"] > 1e-6]

# Define the groups and their specific percentile lists
# Severe gets the 50-100% range, others get the 10-100% range
group_configs = [
    ("Severe (<40)", df3["FEV1%PredST"] < 40, [50, 75, 90]),
    (
        "Moderate (40-69)",
        (df3["FEV1%PredST"] >= 40) & (df3["FEV1%PredST"] < 70),
        [0, 25, 50, 75, 90],
    ),
    ("Mild (>=70)", df3["FEV1%PredST"] >= 70, [0, 25, 50, 75, 90]),
]

# --- Execution ---
for group_name, mask, percentiles in group_configs:
    print(f"\n{'='*40}\nGROUP: {group_name}\n{'='*40}")

    # Isolate the severity group first
    df_group = df3[mask].copy()

    if df_group.empty:
        print("No data in this severity category.")
        continue

    for prctile in percentiles:
        # Calculate the threshold for this specific group's diff column
        # Use (100 - prctile) because if user wants "90% diff",
        # they usually mean the top 10% of differences.
        # If you mean "diff > the 90th percentile value", use (prctile / 100)
        t = df_group[diff_col].abs().quantile(prctile / 100)
        df_sub = df_group[df_group[diff_col].abs() >= t].copy()

        n = len(df_sub)
        print(
            f"\n-- Percentile Threshold: {prctile}% (Threshold value: {t:.2f}, n={n})"
        )

        if n < 5:  # Permutation tests need a tiny bit of data to shuffle
            print("   >>> Skipping: Sample size too small.")
            continue

        # Run Tests
        res_st = perm_test(df_sub["FEV1%PredST"], df_sub[iv_col])
        res_pers = perm_test(df_sub[metric], df_sub[iv_col])

        # Print results with clear formatting
        print(f"   ST Correlation:   r={res_st.statistic:.3f} | p={res_st.pvalue:.4f}")
        print(
            f"   Pers Correlation: r={res_pers.statistic:.3f} | p={res_pers.pvalue:.4f}"
        )


GROUP: Severe (<40)

-- Percentile Threshold: 50% (Threshold value: 0.23, n=155)


/Applications/anaconda3/envs/phd/lib/python3.10/site-packages/scipy/stats/_resampling.py:1492: RuntimeWarning:

overflow encountered in scalar power



   ST Correlation:   r=-0.264 | p=0.0004
   Pers Correlation: r=-0.269 | p=0.0002

-- Percentile Threshold: 75% (Threshold value: 0.32, n=78)
   ST Correlation:   r=-0.236 | p=0.0170
   Pers Correlation: r=-0.242 | p=0.0186

-- Percentile Threshold: 90% (Threshold value: 0.41, n=31)
   ST Correlation:   r=-0.254 | p=0.0872
   Pers Correlation: r=-0.248 | p=0.0882

GROUP: Moderate (40-69)

-- Percentile Threshold: 0% (Threshold value: 0.00, n=735)
   ST Correlation:   r=-0.336 | p=0.0002
   Pers Correlation: r=-0.309 | p=0.0002

-- Percentile Threshold: 25% (Threshold value: 0.44, n=551)
   ST Correlation:   r=-0.292 | p=0.0002
   Pers Correlation: r=-0.250 | p=0.0002

-- Percentile Threshold: 50% (Threshold value: 0.98, n=368)
   ST Correlation:   r=-0.215 | p=0.0002
   Pers Correlation: r=-0.148 | p=0.0022

-- Percentile Threshold: 75% (Threshold value: 1.80, n=184)
   ST Correlation:   r=-0.239 | p=0.0008
   Pers Correlation: r=-0.115 | p=0.0588

-- Percentile Threshold: 90% (Thresho

In [16]:
df[df.Age < 10]

,ID,Age,Height,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,...,FEV1%PredFT,FEV1%PredST,pppFEV1 - ppFEV1ST,ppFEV1FT - ppFEV1ST,P(FEV1|M),"P(FEF2575%FEV1|M, FEV1)","P(bFEV1|M, FEV1, FEF2575%FEV1)",P(D|M),IVs,IV days


# Exploring where model confidently disagrees with baseline

In [9]:
# Compute P(baseline|prediction)
df["P(ppFEV1|AC)"] = vh.calc_P_ppFEV1_given_AC(df, AC)
df["P(ppFEV1|AC ratioed)"] = vh.calc_P_ppFEV1_given_AC(df, AC, corr=True)

# NOTE: AC is undefined above 100%, where ppFEV1 > 100%, value is clipped to 100%

# Plot histogram of P(ppFEV1|AC)
fig = px.histogram(df, x="P(ppFEV1|AC ratioed)", nbins=100)
title = f"Probability of baseline FEV1%pred (clipped) given the predicted airway conductance ratioed"
fig.update_layout(title=title, width=800, height=400)
# fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
fig.show()

In [10]:
# Is the ratioed version much different than the original?
idx_conf_disagree = df[df["P(ppFEV1|AC)"] <= t].index
idx_conf_disagree_ratioed = df[df["P(ppFEV1|AC) ratioed"] <= t_ratioed].index

NameError: name 't' is not defined

## Viz: antibiotics association where the model confidently disagrees with the baseline ppFEV1?

In [13]:
df["P(ppFEV1|AC) ratioed"] = df["P(ppFEV1|AC ratioed)"]

In [4]:
df["clipped ecFEV1%Predicted"] = df["ecFEV1 % Predicted"].clip(upper=100)

import numpy as np
import plotly.graph_objects as go

ivs_col = "Avg Hosp IVs"
ivs_col = "Avg Any antibiotics"

ratioed = True
prctile = 10
df_conf, t = vh.filter_confidently_disagreeing_examples(df, prctile, ratioed)

fig = go.Figure(
    data=go.Scatter(
        x=df_conf["clipped ecFEV1%Predicted"],
        y=df_conf["mean AC"],
        mode="markers",
        marker=dict(
            color=df_conf[ivs_col],
            colorbar=dict(title=ivs_col),
            colorscale="Turbo",
            # colorscale=[
            #     [0.0, "white"],
            #     [[i/(len(px.colors.sequential.Turbo)-1), c] for i, c in enumerate(px.colors.sequential.Turbo)],
            # ],
            # line=dict(
            #     color=np.where(df_conf[ivs_col] == 0, "black", "rgba(0,0,0,0)"),
            #     width=np.where(df_conf[ivs_col] == 0, 1.5, 0),
            # ),
        ),
        hovertemplate="ID: %{customdata[0]}<br>"
        + "clipped ecFEV1%Predicted: %{x}<br>"
        + f"{AC.name} (mean): "
        + "%{y}<br>"
        + f"{ivs_col}: "
        + "%{marker.color}<extra></extra>",
        customdata=np.stack([df_conf["ID"]], axis=-1),
    )
)
# Draw a proportional line (y = x) for reference
min_x = df_conf["clipped ecFEV1%Predicted"].min()
max_x = df_conf["clipped ecFEV1%Predicted"].max()
fig.add_trace(
    go.Scatter(
        x=[min_x, max_x],
        y=[min_x, max_x],
        mode="lines",
        line=dict(color="black", dash="dash"),
    )
)

if ratioed:
    title = f"Scatter plot of ppFEV1 vs mean AC coloured by {ivs_col}<br> P(ppFEV1|AC) {prctile:.0f}th prctile"
else:
    title = f"Scatter plot of ppFEV1 vs mean AC coloured by {ivs_col}<br> P(ppFEV1|AC ratioed) {prctile:.0f}th prctile"

fig.update_layout(
    title=title,
    xaxis_title="FEV1%Predicted (clipped)",
    yaxis_title=AC.name,
    width=800,
    height=700,
    showlegend=False,
)

fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/Antibiotics associations/{title}.pdf")
fig.show()

KeyError: 'P(ppFEV1|AC) ratioed'

In [ ]:
[
    [i / (len(px.colors.sequential.Turbo) - 1), c]
    for i, c in enumerate(px.colors.sequential.Turbo)
]

[[0.0, '#30123b'],
 [0.07142857142857142, '#4145ab'],
 [0.14285714285714285, '#4675ed'],
 [0.21428571428571427, '#39a2fc'],
 [0.2857142857142857, '#1bcfd4'],
 [0.35714285714285715, '#24eca6'],
 [0.42857142857142855, '#61fc6c'],
 [0.5, '#a4fc3b'],
 [0.5714285714285714, '#d1e834'],
 [0.6428571428571429, '#f3c63a'],
 [0.7142857142857143, '#fe9b2d'],
 [0.7857142857142857, '#f36315'],
 [0.8571428571428571, '#d93806'],
 [0.9285714285714286, '#b11901'],
 [1.0, '#7a0402']]

In [15]:
# Superimposed scatter plot of ppFEV1 and AC vs antibiotics

fig = go.Figure()


dftmp = df_conf
# dftmp = df

# Scatter for model predicted mean AC
fig.add_trace(
    go.Scatter(
        x=dftmp["mean AC"],
        y=dftmp["Avg Any antibiotics"],
        mode="markers",
        name="Prediction",
        marker=dict(color="blue"),
        hovertemplate="mean AC: %{x}<br>Avg Any antibiotics: %{y}",
    )
)

# Scatter for baseline (clipped ecFEV1%Predicted)
fig.add_trace(
    go.Scatter(
        x=dftmp["clipped ecFEV1%Predicted"],
        y=dftmp["Avg Any antibiotics"],
        mode="markers",
        name="Baseline",
        marker=dict(color="red"),
        hovertemplate="clipped ecFEV1%Predicted: %{x}<br>Avg Any antibiotics: %{y}",
    )
)

fig.update_traces(marker=dict(size=4))

# Add arrows from baseline prediction (clipped ecFEV1%Predicted) to model output (mean AC)
for ix, row in dftmp.iterrows():
    fig.add_shape(
        type="line",
        x0=row["clipped ecFEV1%Predicted"],
        y0=row["Avg Any antibiotics"],
        x1=row["mean AC"],
        y1=row["Avg Any antibiotics"],
        line=dict(color="gray", width=1, dash="dot"),
        # opacity=1,
        # layer="below"
    )

if ratioed:
    title = f"Superimposed scatter plot of baseline and prediction vs # antibiotics <br> P(ppFEV1|AC ratioed) {prctile:.0f}th prctile"
else:
    title = f"Superimposed scatter plot of baseline and prediction vs # antibiotics <br> P(ppFEV1|AC) {prctile:.0f}th prctile"

fig.update_xaxes(title_text=AC.name)
fig.update_yaxes(title_text="Average number of antibiotics (IV or oral)")
fig.update_layout(
    height=800,
    width=800,
    title=title,
)
# fig.show()
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/Antibiotics associations/{title}.pdf")

## Ranked correlations

In [15]:
df

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,Oral,Non Hosp IVs,Non Hosp IV days,Chest episodes,Cough episodes,Pulm Abscess,Smoking status,2nd hand smoking exposure,IVs,IV days
0,B155916,32,162,1.50,0.47,1.64,Female,2019-01-01,1.50,0.47,...,3.0,1,1,NaN,NaN,NaN,NK,N,3,28.0
1,B155917,45,175,2.67,1.00,2.67,Male,2019-01-01,2.67,1.00,...,2.0,1,1,NaN,NaN,NaN,N,NK,0,0.0
2,B155918,34,191,4.82,3.48,4.99,Male,2019-01-01,4.82,3.48,...,3.0,2,2,NaN,NaN,NaN,N,NK,2,13.0
3,B155921,34,150,1.44,0.59,1.45,Female,2019-01-01,1.44,0.59,...,3.0,0,0,NaN,NaN,NaN,N,NK,2,15.0
4,B155925,38,167,0.92,0.34,1.33,Female,2019-01-01,0.92,0.34,...,6.0,1,1,NaN,NaN,NaN,N,NK,5,61.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2032,C222738,59,158,1.66,1.18,1.95,Female,2019-01-01,1.66,1.18,...,0.0,0,0,NaN,NaN,NaN,N,NK,0,0.0
2033,C222739,69,153,1.63,0.66,1.63,Female,2019-01-01,1.63,0.66,...,6.0,0,0,NaN,NaN,NaN,N,NK,0,0.0
2034,C222741,33,162,1.71,0.86,1.77,Female,2019-01-01,1.71,0.86,...,4.0,0,0,NaN,NaN,NaN,N,NK,3,12.0
2035,C222780,27,176,3.54,4.15,3.54,Female,2019-01-01,3.54,4.15,...,0.0,0,0,NaN,NaN,NaN,N,N,0,0.0


In [ ]:
from scipy.stats import spearmanr

# Spearman correlation
dftmp = df
dftmp = df_conf

# dftmp["AC sampled"] = dftmp[AC.name].apply(lambda ac: AC.sample(n=50, p=ac))

for ab_col in ["Avg Any antibiotics", "Avg IVs", "Avg Oral"]:
    print(ab_col)
    corr_ecfev1, pval_ecfev1 = spearmanr(dftmp["ecFEV1 % Predicted"], dftmp[ab_col])
    print(f"Spearman corr with ppFEV1:  r={corr_ecfev1:.3f}, p={pval_ecfev1:.3g}")

    corr_acsampled, pval_acsampled = spearmanr(dftmp["mean AC"], dftmp[ab_col])
    # corr_acsampled, pval_acsampled = spearmanr(dftmp["AC sampled"].explode(), dftmp.loc[dftmp.index.repeat(50), ab_col].values)
    print(f"Spearman corr with mean AC: r={corr_acsampled:.3f}, p={pval_acsampled:.3g}")

Avg Any antibiotics
Spearman corr with ppFEV1:  r=-0.325, p=2.11e-06
Spearman corr with mean AC: r=-0.306, p=8.44e-06
Avg IVs
Spearman corr with ppFEV1:  r=-0.364, p=8.55e-08
Spearman corr with mean AC: r=-0.342, p=5.34e-07
Avg Oral
Spearman corr with ppFEV1:  r=-0.232, p=0.000849
Spearman corr with mean AC: r=-0.225, p=0.00122


## Dive into large diffs (for day1 2023 - day22019 data)

In [83]:
df_conf.columns

Index(['ID', 'best FEV1', 'Age', 'Height', 'FEV1', 'FEF2575', 'Sex',
       'Date Recorded', 'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1',
       'Predicted FEV1', 'ecFEV1 % Predicted', 'FEV1 % Predicted', 'idx FEV1',
       'idx FEF2575%FEV1', 'idx best FEV1', 'Airway resistance (%)',
       'Airway conductance (%)', 'mean AC', 'P(ppFEV1|AC)',
       'P(ppFEV1|AC) ratioed', 'P(ppFEV1|AC ratioed)',
       'clipped ecFEV1%Predicted', 'Avg Home IVs', 'Avg Hosp IVs', 'Avg IVs',
       'Avg Oral', 'Avg Any antibiotics', 'AC sampled', 'diff (clipped)'],
      dtype='object')

In [ ]:
df_conf["diff (clipped)"] = df_conf["mean AC"] - df_conf["clipped ecFEV1%Predicted"]

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_19058/134932417.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
idx_large_diff = df_conf["P(ppFEV1|AC)"].sort_values(ascending=True).head(20).index
cols2keep = [
    "ID",
    "Sex",
    "Age",
    "Height",
    "FEV1",
    "Predicted FEV1",
    "FEF2575",
    "ecFEF2575%ecFEV1",
    "clipped ecFEV1%Predicted",
    "mean AC",
    "diff (clipped)",
    "P(ppFEV1|AC)",
    "Avg Any antibiotics",
]
df_conf[cols2keep].loc[idx_large_diff].sort_values(by="diff (clipped)", ascending=False)

,ID,Sex,Age,Height,FEV1,Predicted FEV1,FEF2575,ecFEF2575%ecFEV1,clipped ecFEV1%Predicted,mean AC,diff (clipped),P(ppFEV1|AC),Avg Any antibiotics
506,B158714,Female,38,185,2.24,4.031068,0.84,37.499999,55.568404,92.260493,36.692089,1.573554e-14,0.333333
2032,C222546,Female,65,182,1.45,2.992230,0.53,36.551721,48.458848,81.442276,32.983428,1.212774e-07,1.000000
776,B161137,Female,21,169,2.01,3.611874,0.95,47.263681,55.649777,87.670504,32.020726,1.126647e-10,0.600000
1016,B162277,Female,18,183,3.86,4.269905,4.41,114.248704,90.400141,77.712726,-12.687415,1.864928e-10,0.800000
574,B159340,Female,22,176,4.11,3.928017,3.33,81.021893,100.000000,86.646572,-13.353428,6.477634e-07,1.000000
1342,B163943,Female,37,179,3.75,3.786164,3.27,87.199999,99.044839,85.126222,-13.918618,2.134175e-07,2.600000
1831,B169603,Female,30,179,3.35,3.941105,2.57,76.716418,85.001536,71.004430,-13.997106,7.561220e-12,0.500000
708,B160496,Female,19,175,3.69,3.892887,1.58,42.818429,94.788261,80.516846,-14.271415,1.076870e-07,1.000000
288,B157654,Female,41,183,4.15,3.847642,3.42,82.409638,100.000000,85.573716,-14.426284,2.417545e-09,0.600000
2024,C222164,Female,29,166,3.27,3.373554,3.25,99.388380,96.930422,82.371299,-14.559123,1.127210e-08,1.000000


In [ ]:
idx_large_diff = (
    df_conf["diff (clipped)"].abs().sort_values(ascending=False).head(20).index
)
cols2keep = [
    "ID",
    "Sex",
    "Age",
    "Height",
    "FEV1",
    "Predicted FEV1",
    "FEF2575",
    "ecFEF2575%ecFEV1",
    "clipped ecFEV1%Predicted",
    "mean AC",
    "diff (clipped)",
    "P(ppFEV1|AC)",
]
df_conf[cols2keep].loc[idx_large_diff].sort_values(by="diff (clipped)", ascending=False)

,ID,Sex,Age,Height,FEV1,Predicted FEV1,FEF2575,ecFEF2575%ecFEV1,clipped ecFEV1%Predicted,mean AC,diff (clipped),P(ppFEV1|AC)
506,B158714,Female,38,185,2.24,4.031068,0.84,37.499999,55.568404,92.260493,36.692089,1.573554e-14
72,B156254,Female,45,181,1.93,3.632932,4.33,224.352334,53.125137,86.944227,33.819090,3.205089e-06
2032,C222546,Female,65,182,1.45,2.992230,0.53,36.551721,48.458848,81.442276,32.983428,1.212774e-07
776,B161137,Female,21,169,2.01,3.611874,0.95,47.263681,55.649777,87.670504,32.020726,1.126647e-10
219,B157369,Female,26,160,0.89,3.159282,0.40,44.943822,28.170961,56.324774,28.153812,1.041949e-06
508,B158720,Female,30,153,1.57,2.825143,2.56,163.057316,55.572408,79.488189,23.915781,2.249587e-03
1862,B170362,Female,48,172,2.12,3.175287,0.87,41.037738,66.765607,90.127301,23.361693,1.423226e-06
2036,C222614,Female,40,145,1.76,2.367586,1.14,64.772727,74.337311,93.228145,18.890834,2.003711e-03
385,B158151,Female,18,156,1.64,3.043475,2.42,147.560982,53.885768,72.023652,18.137883,6.579071e-03
1800,B169105,Female,37,164,1.37,3.144692,1.71,124.817521,43.565473,61.134284,17.568811,4.848477e-04


In [101]:
df.columns

Index(['ID', 'Date Recorded', 'Airway resistance (%)', 'ecFEV1 % Predicted',
       'Avg Home IVs', 'Avg Hosp IVs', 'Avg IVs', 'Avg Oral',
       'Avg Any antibiotics', 'Airway conductance (%)', 'mean AC',
       'P(ppFEV1|AC)', 'P(ppFEV1|AC ratioed)'],
      dtype='object')

## Computing mean AC vs ppFEV1 diffs

In [ ]:
# diff = Predicted AC - baseline FEV1%pred
df["diff"] = df["mean AC"] - df["ecFEV1 % Predicted"]

df["clipped ecFEV1%Predicted"] = df["ecFEV1 % Predicted"].clip(upper=100)
df["diff (clipped)"] = df["mean AC"] - df["clipped ecFEV1%Predicted"]

In [37]:
df["AC std"] = df[AC.name].apply(lambda ac: AC.get_std(ac))
df["Large diff"] = abs(df["diff (clipped)"]) > df["AC std"]
df["Below 100%"] = df["ecFEV1 % Predicted"] < 100

In [38]:
import plotly.express as px

# Scatter plot with marginal distribution (y axis) for Avg Home IVs
xcol = ""
xcol = " (clipped)"

df_plot = df
# df_plot = df[df["Large diff"]]
# df_plot = df[df["Below 100%"]]

# for col in ["Hosp IVs", "Home IVs", "Any antibiotics", "Oral"]:
for col in ["Any antibiotics"]:
    title = f"Association of model output diff against baseline with {col} (2019-23)"
    fig = px.scatter(
        df_plot,
        x=f"diff{xcol}",
        y=f"Avg {col}",
        labels={
            f"diff{xcol}": f"Predicted conductance - Baseline ppFEV1{xcol}",
            f"Avg {col}": f"Average number of {col}",
        },
        # marginal_y="histogram",
        title=title,
        size_max=6,  # controls the maximum bubble size
        size=[3] * len(df_plot),
        hover_data=["ID"],  # Add this line to include df.ID in hover label
    )
    # fig.update_traces(marker=dict(size=3))
    fig.update_layout(height=600, width=800)
    fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")

In [ ]:
# Why so many individuals with worse lungs have no hosp IVs?

# Home and hops IVs have a similar pattern

# Inaccuracies
# 1. Deal with individuals that have FEV1% > 100%

# Biases
# More people with negative diff
# 2. Few individuals have 3+ IVs

# TODO
# 1. Show marked difference by excluding IDs when baseline is within 1 sigma from prediction
# 2. Compute percentage of people per number of IVs?
# Verify the pattern on other years
# Plot home/hosp IVs between 2019 and 2023. Max 20 per year. Check IV data completeness over the years, if missing values then compute an average number

In [ ]:
# Scatter plot with marginal distribution (y axis) for Avg Home IVs
xcol = ""
xcol = " (clipped)"

df_plot = df
# df_plot = df[df["Large diff"]]
# df_plot = df[df["Below 100%"]]

# Create the three dataframes
df_mild = df_plot[df_plot["mean AC"] >= 70]
df_moderate = df_plot[(df_plot["mean AC"] >= 40) & (df_plot["mean AC"] < 70)]
df_severe = df_plot[(df_plot["mean AC"] < 40)]

for col in ["Hosp"]:  # , "Home"]:
    title = f"Association of model output diff against baseline with {col} IVs (2019-23) - large diff"
    fig = make_subplots(rows=1, cols=3)

    for i, dftmp in enumerate([df_mild, df_moderate, df_severe]):
        fig.add_trace(
            go.Scatter(
                x=dftmp[f"diff{xcol}"],
                y=dftmp[f"Avg {col} IVs"],
                mode="markers",
                marker=dict(color="#0072b2", size=3),
            ),
            row=1,
            col=i + 1,
        )

    # fig.update_traces(marker=dict(size=3))
    fig.update_layout(height=500, width=1000)
    # fig.show()
    fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/Plot 1.pdf")

In [54]:
from scipy.stats import pearsonr

# Compute correlation between 'ecFEV1 % Predicted' and 'Avg Hosp IVs'
x1 = df["ecFEV1 % Predicted"]
y1 = df["Avg Hosp IVs"]
corr1, pval1 = pearsonr(x1, y1)

# Compute correlation between 'mean AC' and 'Avg Hosp IVs'
x2 = df["mean AC"]
y2 = df["Avg Hosp IVs"]
corr2, pval2 = pearsonr(x2, y2)

print(f"corr1, pval1: {corr1:.3f}, {pval1:.3g}")
print(f"corr2, pval2: {corr2:.3f}, {pval2:.3g}")

corr1, pval1: -0.376, 1.53e-69
corr2, pval2: -0.380, 2.22e-71


In [ ]:
import plotly.express as px
import numpy as np

# Scatter plot with marginal distribution (y axis) for Avg Home IVs
xcol = ""
xcol = " (clipped)"

df_plot = df
# df_plot = df[df["Large diff"]]
# df_plot = df[df["Below 100%"]]

for col in ["Hosp", "Home"]:
    title = f"Association of model output diff against baseline with {col} IVs (2019-23) - large diff"

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df_plot["clipped ecFEV1%Predicted"],
            y=df_plot[f"Avg {col} IVs"],
            mode="markers",
        )
    )
    fig.add_trace(
        go.Scatter(x=df_plot["mean AC"], y=df_plot[f"Avg {col} IVs"], mode="markers")
    )

    fig.update_traces(marker=dict(size=3))
    fig.update_layout(height=800, width=800)
    fig.show()
    # fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")